# Minimum-overlap MRI triplets, 384×384

Same six-slot selection and pixel/label preprocessing as the previous series-safe dataset. Up to 60 unique same-series triplets, up to 180 distinct source slices. Exact allocation minimizes repeated slice uses. Retain original 18/14/12/8/12/8 percentile references. Gold74 held out. No training is performed.


In [ ]:
BASE_HELPER = '"""Six MRI slots with the saved Raptor V5 image preprocessing contract.\n\nRuns on Kaggle. Outputs fully normalized float16 [K,3,384,384] study bags.\nNo DICOM data is downloaded to the local development computer.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport io\nimport json\nimport os\nimport re\nimport shutil\nimport threading\nimport time\nimport zipfile\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport pydicom\nimport torch\nimport torch.nn.functional as F\nfrom pydicom.pixels import apply_modality_lut\n\n# Six-slot extension; original V5 used five slots with 18/14/12/8/12 references.\nSLOTS = [\n    ("SAG_FLUID_FS", "Sagittal", 1, 1, 18),\n    ("COR_FLUID_FS", "Coronal", 1, 1, 12),\n    ("AX_FLUID_FS", "Axial", 1, 1, 12),\n    ("SAG_FLUID_NOFS", "Sagittal", 1, 0, 14),\n    ("COR_T1", "Coronal", 0, 0, 8),\n    ("SAG_T1", "Sagittal", 0, 0, 8),\n]\nMEAN = torch.tensor([.485, .456, .406]).view(1, 3, 1, 1)\nSTD = torch.tensor([.229, .224, .225]).view(1, 3, 1, 1)\nLABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",\n          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\nRECIPE = {\n    "format": "rsna_six_slot_raptor_v5_v1", "slots": SLOTS,\n    "source_span": [.02, .98], "intensity_percentiles": [2, 98],\n    "intensity_reference": "original evenly spaced per-series reference slices",\n    "crop_mm": 140, "cache_resolution": 336, "model_resolution": 384,\n    "resize": "cv2.INTER_AREA to 336; torch bilinear align_corners=False to 384",\n    "quantization": "uint8 after 336 resize, before /255 and 384 interpolation",\n    "mean": [.485, .456, .406], "std": [.229, .224, .225],\n    "dtype": "float16", "normalized": True, "source_budget": 64,\n    "triplets": "sliding adjacent sampled slices in concatenated slot order, like parent V5",\n    "missing_slots": "absent, never substitute a different category or fabricate pixels",\n    "selector": "dicom_weight_type; positive fat-sat header evidence overrides metadata",\n    "labels": "annotations are separate from pixels; gold74 excluded from training split",\n}\nRECIPE_SHA = hashlib.sha256(json.dumps(RECIPE, sort_keys=True).encode()).hexdigest()\nLOCK = threading.Lock()\n\n\ndef json_write(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n    tmp.replace(path)\n\n\ndef allocate(capacities, weights, target=64):\n    capacities = np.asarray(capacities, dtype=np.int64)\n    weights = np.asarray(weights, dtype=np.float64)\n    ideal = target * weights / weights.sum()\n    quotas = np.floor(ideal).astype(np.int64)\n    for i in np.argsort(-(ideal-quotas), kind="stable")[:target-int(quotas.sum())]:\n        quotas[i] += 1\n    quotas = np.minimum(quotas, capacities)\n    deficit = np.zeros(len(quotas))\n    wanted = min(target, int(capacities.sum()))\n    while quotas.sum() < wanted:\n        available = quotas < capacities\n        active = np.where(available, weights, 0)\n        deficit[available] += active[available]/active.sum()\n        choice = int(np.argmax(np.where(available, deficit, -np.inf)))\n        quotas[choice] += 1\n        deficit[choice] -= 1\n    return quotas\n\n\ndef flag(value):\n    try:\n        value = float(value)\n        return int(value) if value in (0, 1) else None\n    except (ValueError, TypeError):\n        return None\n\n\ndef weight_type(h):\n    desc = re.sub(r"[_\\-.]", " ", f"{getattr(h, \'SeriesDescription\', \'\')} {getattr(h, \'SequenceName\', \'\')}".lower())\n    t1 = bool(re.search(r"\\bt1\\b|\\bt1w\\b", desc))\n    t2 = bool(re.search(r"\\bt2\\b|\\bt2w\\b", desc))\n    pdw = bool(re.search(r"\\bpd\\b|\\bpdw\\b|proton|\\bdp\\b|dens", desc))\n    tr, te = float(getattr(h, \'RepetitionTime\', \'nan\')), float(getattr(h, \'EchoTime\', \'nan\'))\n    if t1 and not t2 and not pdw:\n        return "T1"\n    if t2 and not pdw:\n        return "T2"\n    if pdw:\n        return "PD"\n    if "GR" in str(getattr(h, "ScanningSequence", "")).upper():\n        return "GRE"\n    return "T1" if tr < 800 else "T2" if te > 60 else "PD" if tr >= 800 else "UNK"\n\n\ndef classify(row, directory):\n    files = sorted(directory.glob("*.dcm"))\n    if not files:\n        return None\n    h = pydicom.dcmread(files[len(files)//2], stop_before_pixels=True)\n    kind = weight_type(h)\n    # In this competition CSV, Fluid_Sensitive and Fat_Suppression are\n    # effectively coupled. PD/T2 without FS must be recovered from the header,\n    # as in the original recovered six-slot classifier.\n    fluid = int(kind in ("PD", "T2")) if kind != "UNK" else flag(row.get("Fluid_Sensitive"))\n    fs = flag(row.get("Fat_Suppression"))\n    desc = re.sub(r"[_\\-.]", " ", f"{getattr(h, \'SeriesDescription\', \'\')} {getattr(h, \'SequenceName\', \'\')}".lower())\n    options = getattr(h, "ScanOptions", [])\n    options = options.split(\'\\\\\') if isinstance(options, str) else options\n    opts = set(str(v).upper() for v in options)\n    header_fs = bool(re.search(r"\\bfs\\b|fat\\s*sat|\\bstir\\b|\\bspair\\b|\\bspir\\b|\\bwe\\b|water excit|\\btirm\\b|\\bsting\\b|\\bfatsup\\b|smart fat|\\bwater\\b", desc)) or bool(opts & {"FS", "FATSAT", "FAT_SAT", "FSAT"})\n    if header_fs:\n        fs = 1\n    elif fs is None:\n        fs = 0\n    return {"uid": str(row["SeriesInstanceUID"]), "plane": row["Anatomical_Plane"],\n            "fluid": fluid, "fs": fs, "weight": kind, "files": files,\n            "description": str(getattr(h, "SeriesDescription", ""))}\n\n\ndef order_files(series):\n    records, sop_uids = [], set()\n    for file in series["files"]:\n        h = pydicom.dcmread(file, stop_before_pixels=True)\n        sop = str(h.SOPInstanceUID)\n        if sop in sop_uids:\n            raise ValueError(f"Duplicate SOPInstanceUID: {file}")\n        sop_uids.add(sop)\n        if str(h.SeriesInstanceUID) != series["uid"]:\n            raise ValueError(f"Series UID mismatch: {file}")\n        iop, ipp = getattr(h, "ImageOrientationPatient", None), getattr(h, "ImagePositionPatient", None)\n        if iop is not None and ipp is not None and len(iop) == 6:\n            position = float(np.dot(np.asarray(ipp, float), np.cross(np.asarray(iop[:3], float), np.asarray(iop[3:], float))))\n        else:\n            position = float(h.InstanceNumber)\n        if not np.isfinite(position):\n            raise ValueError(f"Nonfinite slice coordinate: {file}")\n        spacing = float(getattr(h, "PixelSpacing", [.5])[0])\n        records.append((position, str(file), spacing))\n    records.sort(key=lambda r: (r[0], r[1]))\n    good_spacing = [r[2] for r in records if np.isfinite(r[2]) and r[2] > 0]\n    median = float(np.median(good_spacing)) if good_spacing else .5\n    return [(file, spacing if np.isfinite(spacing) and spacing > 0 else median) for _, file, spacing in records]\n\n\ndef pixels(file):\n    d = pydicom.dcmread(file)\n    a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n    if a.ndim != 2 or not np.isfinite(a).all():\n        raise ValueError(f"Invalid acquired pixels: {file}")\n    if str(getattr(d, "PhotometricInterpretation", "")) == "MONOCHROME1":\n        a = a.max()-a\n    return a\n\n\ndef crop_plane(a, spacing, low, high):\n    a = np.clip((a-low)/(high-low+1e-6), 0, 1)\n    h, w = a.shape\n    size = min(int(round(140/max(spacing, .001))), min(h, w))\n    y, x = (h-size)//2, (w-size)//2\n    a = cv2.resize(a[y:y+size, x:x+size], (336, 336), interpolation=cv2.INTER_AREA)\n    return (a*255).astype(np.uint8)\n\n\ndef model_ready(planes):\n    # Preserve parent\'s /255, interpolation, then channel-specific normalization.\n    volume = np.stack(planes)\n    source = torch.from_numpy(volume.astype(np.float32)/255)\n    resized = F.interpolate(source[:, None], size=(384, 384), mode="bilinear", align_corners=False)[:, 0]\n    x = torch.stack([resized[:-2], resized[1:-1], resized[2:]], dim=1)\n    return ((x-MEAN)/STD).numpy().astype(np.float16)\n\n\ndef prepare(uid, rows, series_root):\n    candidates, warnings = [], []\n    for row in rows:\n        s = classify(row, series_root/uid/str(row["SeriesInstanceUID"]))\n        if s is not None:\n            candidates.append(s)\n    used, pools, selection = set(), [], []\n    for name, plane, fluid, fs, reference_count in SLOTS:\n        matching = [s for s in candidates if s["uid"] not in used and s["plane"] == plane\n                    and s["fluid"] == fluid and s["fs"] == fs\n                    and (fluid == 1 or s["weight"] == "T1")]\n        # Retain metadata row priority, matching the Raptor selection convention.\n        if not matching:\n            pools.append(None)\n            selection.append({"slot": name, "present": False})\n            continue\n        series = matching[0]\n        used.add(series["uid"])\n        files = order_files(series)\n        lo = int(len(files)*.02)\n        hi = max(int(len(files)*.98)-1, lo)\n        refs = np.linspace(lo, hi, reference_count).round().astype(int)\n        pools.append((files, lo, hi, refs))\n        selection.append({"slot": name, "present": True, "series_uid": series["uid"],\n                          "description": series["description"], "weight": series["weight"]})\n    capacities = [0 if p is None else p[2]-p[1]+1 for p in pools]\n    quotas = allocate(capacities, [s[4] for s in SLOTS])\n    planes, slot_ids, sources, intensity = [], [], [], []\n    for slot, (pool, count) in enumerate(zip(pools, quotas)):\n        if pool is None or not count:\n            continue\n        files, lo, hi, refs = pool\n        picks = lo + np.linspace(0, hi-lo, int(count)).round().astype(int)\n        arrays = {int(i): pixels(files[int(i)][0]) for i in sorted(set(refs)|set(picks))}\n        low, high = np.percentile(np.concatenate([arrays[int(i)].ravel() for i in refs]), [2, 98])\n        intensity.append({"slot": slot, "p2": float(low), "p98": float(high), "reference_indices": refs.tolist()})\n        for i in picks:\n            file, spacing = files[int(i)]\n            planes.append(crop_plane(arrays[int(i)], spacing, low, high))\n            slot_ids.append(slot)\n            sources.append({"slot": slot, "file": Path(file).name, "series_uid": selection[slot]["series_uid"], "native_index": int(i)})\n    if len(planes) < 3:\n        raise ValueError("Fewer than three acquired slices across the six categories")\n    x = model_ready(planes)\n    ids = np.asarray(slot_ids, dtype=np.int8)\n    triplet_slots = np.stack([ids[:-2], ids[1:-1], ids[2:]], 1)\n    pure = (triplet_slots[:, 0] == triplet_slots[:, 1]) & (triplet_slots[:, 1] == triplet_slots[:, 2])\n    audit = {"study_uid": uid, "recipe_sha256": RECIPE_SHA, "selection": selection,\n             "capacities": capacities, "quotas": quotas.tolist(), "source_slices": len(planes),\n             "windows": len(x), "cross_series_windows": int((~pure).sum()),\n             "source_files": sources, "intensity": intensity, "warnings": warnings}\n    if not np.isfinite(x).all() or tuple(x.shape[1:]) != (3, 384, 384):\n        raise ValueError("Nonfinite or invalid model input")\n    return {"images": x, "triplet_slot_ids": triplet_slots, "same_series": pure,\n            "slot_present": np.asarray([s["present"] for s in selection], bool),\n            "audit_json": np.asarray(json.dumps(audit, allow_nan=False))}, audit\n\n\ndef save_npz(path, data):\n    tmp = path.with_suffix(".npz.tmp")\n    with zipfile.ZipFile(tmp, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:\n        for name, array in data.items():\n            with z.open(name+".npy", "w", force_zip64=True) as member:\n                np.lib.format.write_array(member, np.asarray(array), allow_pickle=False)\n    tmp.replace(path)\n\n\ndef find_root():\n    for p in ("/kaggle/input/competitions/rsna-knee-abnormality-detection", "/kaggle/input/rsna-knee-abnormality-detection"):\n        if (Path(p)/"train_series.csv").is_file():\n            return Path(p)\n    raise FileNotFoundError("Attach rsna-knee-abnormality-detection competition data")\n\n\ndef run(shard=0, shards=32, limit=None, root=None, output=None, workers=2):\n    print(json.dumps({\'phase\':\'input_setup\',\'shard\':shard}), flush=True)\n    torch.set_num_threads(1)\n    cv2.setNumThreads(1)\n    root = find_root() if root is None else Path(root)\n    output = Path(output or f"/kaggle/working/six_slot_raptor_shard_{shard:02d}")\n    output.mkdir(parents=True, exist_ok=True)\n    series = pd.read_csv(root/"train_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})\n    labels = pd.read_csv(root/"train.csv", dtype={"StudyInstanceUID": str})\n    ids = sorted(labels.StudyInstanceUID.unique())[shard::shards]\n    if limit:\n        ids = ids[:limit]\n    groups = {str(uid): g.to_dict("records") for uid, g in series.groupby("StudyInstanceUID", sort=False)}\n    series_root = next((root/p for p in ("train_series", "train_images", "train") if (root/p).is_dir()), None)\n    if series_root is None:\n        raise FileNotFoundError("No competition training DICOM directory")\n    json_write(output/"recipe.json", dict(RECIPE, sha256=RECIPE_SHA))\n    labels[labels.StudyInstanceUID.isin(ids)].to_csv(output/"labels_official.csv", index=False)\n    gold = set(labels.loc[labels[LABELS].notna().all(axis=1), "StudyInstanceUID"])\n    # Never recursively scan the competition\'s millions of DICOM files to find\n    # a small annotation CSV; use the two supported attached-dataset layouts.\n    jev = next((p/\'gold74_eval_set.csv\' for p in (\n        Path(\'/kaggle/input/datasets/riadmohamed42/jev-knee-labels\'),\n        Path(\'/kaggle/input/jev-knee-labels\')) if (p/\'gold74_eval_set.csv\').is_file()), None)\n    if jev is None:\n        raise FileNotFoundError(\'Attach riadmohamed42/jev-knee-labels: gold74 metadata is required\')\n    if jev is not None:\n        gold |= set(pd.read_csv(jev, dtype={"StudyInstanceUID": str}).StudyInstanceUID)\n        final2 = jev.parent/"FINAL2_labels_scores.csv"\n        if final2.is_file():\n            table = pd.read_csv(final2, dtype={"StudyInstanceUID": str})\n            table[table.StudyInstanceUID.isin(ids)].to_csv(output/"labels_jev_final2.csv", index=False)\n    pd.DataFrame({"StudyInstanceUID": ids, "split": ["held_out_gold" if uid in gold else "train" for uid in ids]}).to_csv(output/"splits.csv", index=False)\n    print(json.dumps({\'phase\':\'metadata_ready\',\'shard\':shard,\'studies\':len(ids),\'series_root\':str(series_root)}), flush=True)\n    start = time.time()\n    results, failures = [], []\n    def heartbeat(state="running"):\n        value = {"state": state, "shard": shard, "shards": shards, "expected_studies": len(ids),\n                 "completed_studies": len(results), "failed_studies": len(failures),\n                 "elapsed_seconds": round(time.time()-start, 1), "updated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n                 "output_bytes": sum(r["bytes"] for r in results), "recipe_sha256": RECIPE_SHA}\n        json_write(output/"progress.json", value)\n        print(json.dumps(value), flush=True)\n    def one(uid):\n        file = output/(uid+".npz")\n        # Reserve worst-case remaining uncompressed bag size before processing.\n        if shutil.disk_usage(output).free < 512*1024**2:\n            raise OSError("Less than 512 MiB free: stop before corrupting output")\n        data, audit = prepare(uid, groups[uid], series_root)\n        save_npz(file, data)\n        # Re-open and check the exact saved model tensor.\n        with np.load(file, allow_pickle=False) as saved:\n            if not np.array_equal(saved["images"], data["images"]):\n                raise ValueError("Saved tensor round-trip mismatch")\n        digest = hashlib.sha256()\n        with file.open("rb") as f:\n            for block in iter(lambda: f.read(8*1024**2), b""):\n                digest.update(block)\n        return {"study_uid": uid, "file": file.name, "windows": audit["windows"],\n                "cross_series_windows": audit["cross_series_windows"], "slot_present": data["slot_present"].tolist(),\n                "bytes": file.stat().st_size, "sha256": digest.hexdigest()}\n    heartbeat()\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        futures = {pool.submit(one, uid): uid for uid in ids}\n        for future in as_completed(futures):\n            uid = futures[future]\n            try:\n                row = future.result()\n                results.append(row)\n                with (output/"index.jsonl").open("a") as f:\n                    f.write(json.dumps(row)+"\\n")\n            except Exception as exc:\n                failure = {"study_uid": uid, "error": f"{type(exc).__name__}: {exc}"}\n                failures.append(failure)\n                with (output/"failures.jsonl").open("a") as f:\n                    f.write(json.dumps(failure)+"\\n")\n                print("STUDY_FAILED "+json.dumps(failure), flush=True)\n            heartbeat()\n    manifest = {"recipe_sha256": RECIPE_SHA, "shard": shard, "shards": shards,\n                "expected_uids": ids, "completed": results, "failures": failures,\n                "complete": len(results) == len(ids) and not failures}\n    json_write(output/"manifest.json", manifest)\n    heartbeat("complete" if manifest["complete"] else "incomplete")\n    if failures:\n        raise RuntimeError(f"Shard incomplete: {len(failures)} failed studies; see failures.jsonl")\n    return manifest\n\n\n'
PARENT_HELPER = '"""Six-slot direct-384 exports with provably single-series triplets."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport shutil\nimport time\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\n\nfrom preprocess_six_slot_raptor import classify, pixels, save_npz, find_root, json_write\nimport pydicom\n\nSLOTS = [\n    (\'SAG_FLUID_FS\',\'Sagittal\',1,1,18),\n    (\'SAG_FLUID_NOFS\',\'Sagittal\',1,0,14),\n    (\'COR_FLUID_FS\',\'Coronal\',1,1,12),\n    (\'COR_T1\',\'Coronal\',0,0,8),\n    (\'AX_FLUID_FS\',\'Axial\',1,1,12),\n    (\'SAG_T1\',\'Sagittal\',0,0,8),\n]\nLABELS = [\'ACL\',\'MCL\',\'Medial Meniscus\',\'Lateral Meniscus\',\'Medial OA\',\'Lateral OA\',\n          \'PF OA\',\'Effusion\',\'Synovitis\',"Baker\'s",\'Contusion\',\'Fracture\']\nMEAN = np.asarray([.485,.456,.406],np.float32).reshape(1,3,1,1)\nSTD = np.asarray([.229,.224,.225],np.float32).reshape(1,3,1,1)\nRECIPE = dict(format=\'rsna_series_safe_384_v1\',slots=SLOTS,source_span=[.02,.98],\n    depth_indices=\'lo=floor(N*.02); hi=max(floor(N*.98)-1,lo), inclusive\',\n    intensity_percentiles=[2,98],intensity_reference=\'the actual unique sampled slices in each selected series\',\n    sampling=\'evenly spaced unique physical-order slices, up to slot allocation; no redistribution or duplication\',\n    source_budget=72,crop_mm=140,crop=\'center rectangle using row/column PixelSpacing separately, clamp to acquired field\',\n    resolution=384,model_resolution=384,cache_resolution=384,resize=\'direct cv2.INTER_AREA to 384; no intermediate resize\',\n    quantization=\'floor(clipped_scaled_resized*255) uint8\',slices_dtype=\'uint8\',dtype=\'float16\',normalized=True,\n    mean=[.485,.456,.406],std=[.229,.224,.225],triplets=\'sliding three neighboring sampled slices WITHIN one SeriesInstanceUID\',\n    missing_slots=\'absent; never substitute, pad or duplicate slices\',selector=\'dicom_weight_type; positive fat-sat header evidence overrides metadata\',\n    physical_order=\'common IOP slice normal dot IPP; fail on missing/inconsistent geometry\',\n    labels=\'Gold74 excluded from training split; held_out_gold separate; HYBRID scores separate from pixels\')\nRECIPE_SHA=hashlib.sha256(json.dumps(RECIPE,sort_keys=True).encode()).hexdigest()\n\n\ndef physical_order(series,study_uid):\n    records=[]; sop_uids=set(); normal=None\n    for file in series[\'files\']:\n        h=pydicom.dcmread(file,stop_before_pixels=True)\n        if str(h.SeriesInstanceUID)!=series[\'uid\'] or str(h.StudyInstanceUID)!=study_uid:\n            raise ValueError(\'DICOM series/study ownership mismatch\')\n        sop=str(h.SOPInstanceUID)\n        if sop in sop_uids:raise ValueError(\'Duplicate acquired SOPInstanceUID\')\n        sop_uids.add(sop)\n        iop=np.asarray(getattr(h,\'ImageOrientationPatient\',[]),float)\n        ipp=np.asarray(getattr(h,\'ImagePositionPatient\',[]),float)\n        spacing=np.asarray(getattr(h,\'PixelSpacing\',[]),float)\n        if iop.shape!=(6,) or ipp.shape!=(3,) or spacing.shape!=(2,) or not np.isfinite(np.r_[iop,ipp,spacing]).all() or (spacing<=0).any():\n            raise ValueError(\'Missing/invalid physical geometry or PixelSpacing\')\n        candidate=np.cross(iop[:3],iop[3:])\n        if np.linalg.norm(candidate)<.99:raise ValueError(\'Invalid slice normal\')\n        candidate=candidate/np.linalg.norm(candidate)\n        if normal is None:normal=candidate\n        if np.dot(normal,candidate)<.999:raise ValueError(\'Inconsistent slice orientations\')\n        records.append(dict(file=str(file),spacing=spacing.tolist(),position=float(np.dot(ipp,normal)),sop_uid=sop))\n    records.sort(key=lambda r:(r[\'position\'],r[\'sop_uid\']))\n    if len({round(r[\'position\'],5) for r in records})!=len(records):\n        raise ValueError(\'Duplicate physical slice positions\')\n    return records\n\n\ndef sample_indices(count,allocation):\n    if count<1:raise ValueError(\'Empty MRI series\')\n    lo=int(count*.02); hi=max(int(count*.98)-1,lo)\n    chosen=np.linspace(lo,hi,min(allocation,hi-lo+1)).round().astype(np.int32)\n    if len(set(chosen.tolist()))!=len(chosen):raise ValueError(\'Duplicate sampled index\')\n    return chosen\n\n\ndef crop_plane(array,spacing,low,high):\n    scaled=np.clip((array-low)/(high-low+1e-6),0,1)\n    height,width=scaled.shape\n    ch=min(max(1,int(round(140/spacing[0]))),height)\n    cw=min(max(1,int(round(140/spacing[1]))),width)\n    top,left=(height-ch)//2,(width-cw)//2\n    resized=cv2.resize(scaled[top:top+ch,left:left+cw],(384,384),interpolation=cv2.INTER_AREA)\n    return (resized*255).astype(np.uint8)\n\n\ndef normalized_triplets(slices,indices):\n    x=slices[indices].astype(np.float32)/np.float32(255.)\n    return ((x-MEAN)/STD).astype(np.float16)\n\n\ndef validate_arrays(data):\n    slices,indices,x=data[\'slices_uint8\'],data[\'triplet_indices\'],data[\'images\']\n    if slices.dtype!=np.uint8 or slices.ndim!=3 or slices.shape[1:]!=(384,384):raise ValueError(\'Invalid stored slices\')\n    if not len(indices) or indices.shape!=(len(x),3) or (indices<0).any() or (indices>=len(slices)).any():raise ValueError(\'Invalid triplet indices\')\n    owners=data[\'slice_series_uids\'][indices]\n    if not np.all(owners==owners[:,:1]):raise ValueError(\'CROSS-SERIES TRIPLET\')\n    if not np.all(data[\'same_series\']) or not np.all(data[\'triplet_slot_ids\']==data[\'slice_slot_ids\'][indices]):raise ValueError(\'Triplet provenance mismatch\')\n    if not np.all(owners[:,0]==data[\'triplet_series_uids\']):raise ValueError(\'Triplet series UID mismatch\')\n    if x.dtype!=np.float16 or x.shape[1:]!=(3,384,384) or not np.isfinite(x).all():raise ValueError(\'Invalid normalized images\')\n    if not np.array_equal(normalized_triplets(slices,indices),x):raise ValueError(\'Normalized images differ from stored slices\')\n\n\ndef prepare(uid,rows,series_root):\n    candidates=[]\n    for row in rows:\n        candidate=classify(row,series_root/uid/str(row[\'SeriesInstanceUID\']))\n        if candidate is not None:candidates.append(candidate)\n    used=set(); selection=[]; slices=[]; sources=[]; slot_ids=[]; series_uids=[]; triplets=[]; intensity=[]\n    for slot,(name,plane,fluid,fs,allocation) in enumerate(SLOTS):\n        matches=[s for s in candidates if s[\'uid\'] not in used and s[\'plane\']==plane and\n                 s[\'fluid\']==fluid and s[\'fs\']==fs and (fluid or s[\'weight\']==\'T1\')]\n        if not matches:\n            selection.append(dict(slot=name,present=False));continue\n        series=matches[0];used.add(series[\'uid\'])\n        files=physical_order(series,uid); picks=sample_indices(len(files),allocation)\n        arrays=[pixels(files[int(i)][\'file\']) for i in picks]\n        low,high=np.percentile(np.concatenate([a.ravel() for a in arrays]),[2,98])\n        offset=len(slices)\n        for native_index,array in zip(picks,arrays):\n            record=files[int(native_index)]\n            slices.append(crop_plane(array,record[\'spacing\'],low,high))\n            slot_ids.append(slot);series_uids.append(series[\'uid\'])\n            sources.append(dict(slot=slot,series_uid=series[\'uid\'],file=Path(record[\'file\']).name,\n                                sop_uid=record[\'sop_uid\'],native_index=int(native_index),position=record[\'position\'],spacing=record[\'spacing\']))\n        triplets.extend([[offset+i,offset+i+1,offset+i+2] for i in range(max(0,len(picks)-2))])\n        selection.append(dict(slot=name,present=True,series_uid=series[\'uid\'],description=series[\'description\'],\n                              weight=series[\'weight\'],allocation=allocation,sampled_count=len(picks)))\n        intensity.append(dict(slot=slot,p2=float(low),p98=float(high),reference_indices=picks.tolist()))\n    if not triplets:raise ValueError(\'No series has three eligible sampled slices\')\n    slices=np.stack(slices); indices=np.asarray(triplets,np.int32); slot_ids=np.asarray(slot_ids,np.int8)\n    series_uids=np.asarray(series_uids)\n    x=normalized_triplets(slices,indices)\n    audit=dict(study_uid=uid,recipe_sha256=RECIPE_SHA,selection=selection,source_slices=len(slices),windows=len(x),\n               cross_series_windows=0,source_files=sources,intensity=intensity)\n    data=dict(slices_uint8=slices,slice_slot_ids=slot_ids,slice_series_uids=series_uids,\n              triplet_indices=indices,triplet_series_uids=series_uids[indices[:,0]],\n              images=x,triplet_slot_ids=slot_ids[indices],same_series=np.ones(len(x),bool),\n              slot_present=np.asarray([s[\'present\'] for s in selection],bool),audit_json=np.asarray(json.dumps(audit,allow_nan=False)))\n    validate_arrays(data)\n    return data,audit\n\n\ndef run(shard=0,shards=32,limit=None,root=None,output=None,workers=4):\n    torch.set_num_threads(1);cv2.setNumThreads(1)\n    root=Path(root) if root else find_root()\n    output=Path(output or f\'/kaggle/working/series_safe_384_shard_{shard:02d}\')\n    output.mkdir(parents=True,exist_ok=True)\n    series=pd.read_csv(root/\'train_series.csv\',dtype={\'StudyInstanceUID\':str,\'SeriesInstanceUID\':str})\n    labels=pd.read_csv(root/\'train.csv\',dtype={\'StudyInstanceUID\':str})\n    jev=next((p for p in [Path(\'/kaggle/input/datasets/riadmohamed42/jev-knee-labels\'),Path(\'/kaggle/input/jev-knee-labels\')]\n              if (p/\'gold74_eval_set.csv\').is_file()),None)\n    if jev is None:raise FileNotFoundError(\'Gold74 exclusion requires riadmohamed42/jev-knee-labels\')\n    gold_labels=pd.read_csv(jev/\'gold74_eval_set.csv\',dtype={\'StudyInstanceUID\':str})\n    gold=set(gold_labels.StudyInstanceUID)\n    official_gold=set(labels.loc[labels[LABELS].notna().all(axis=1),\'StudyInstanceUID\'])\n    if len(gold)!=74 or not official_gold<=gold:raise ValueError(\'Gold74 metadata mismatch\')\n    all_ids=sorted(labels.StudyInstanceUID.unique());ids=all_ids[shard::shards]\n    if limit is not None:ids=ids[:limit]\n    if not set(gold)<=set(all_ids):raise ValueError(\'Gold74 IDs outside competition corpus\')\n    groups={str(uid):group.to_dict(\'records\') for uid,group in series.groupby(\'StudyInstanceUID\',sort=False)}\n    series_root=next((root/p for p in (\'train_series\',\'train_images\',\'train\') if (root/p).is_dir()),None)\n    if series_root is None:raise FileNotFoundError(\'Training DICOM directory not found\')\n    json_write(output/\'recipe.json\',{**RECIPE,\'sha256\':RECIPE_SHA})\n    split=pd.DataFrame(dict(StudyInstanceUID=ids,split=[\'held_out_gold\' if uid in gold else \'train\' for uid in ids]))\n    split.to_csv(output/\'splits.csv\',index=False)\n    labels[labels.StudyInstanceUID.isin(ids)].to_csv(output/\'labels_official.csv\',index=False)\n    gold_labels[gold_labels.StudyInstanceUID.isin(ids)].to_csv(output/\'labels_gold74.csv\',index=False)\n    train_ids=set(ids)-gold\n    hybrid=pd.read_csv(jev/\'HYBRID_labels_scores.csv\',dtype={\'StudyInstanceUID\':str})\n    hybrid=hybrid[hybrid.StudyInstanceUID.isin(train_ids)]\n    if set(hybrid.StudyInstanceUID)!=train_ids or hybrid.StudyInstanceUID.duplicated().any() or not np.isfinite(hybrid[LABELS].to_numpy()).all():\n        raise ValueError(\'Missing or invalid non-gold HYBRID targets\')\n    hybrid.to_csv(output/\'labels_hybrid_train.csv\',index=False)\n    results=[];failures=[];start=time.time()\n    def progress(state=\'running\'):\n        record=dict(state=state,shard=shard,shards=shards,expected_studies=len(ids),completed_studies=len(results),\n                    failed_studies=len(failures),output_bytes=sum(r[\'bytes\'] for r in results),\n                    elapsed_seconds=round(time.time()-start,1),recipe_sha256=RECIPE_SHA,\n                    updated_utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\',time.gmtime()))\n        json_write(output/\'progress.json\',record);print(json.dumps(record),flush=True)\n    def one(uid):\n        if shutil.disk_usage(output).free<512*1024**2:raise OSError(\'Insufficient disk space\')\n        data,audit=prepare(uid,groups[uid],series_root)\n        split_name=\'held_out_gold\' if uid in gold else \'train\'\n        folder=output/split_name;folder.mkdir(exist_ok=True)\n        file=folder/(uid+\'.npz\');save_npz(file,data)\n        with np.load(file,allow_pickle=False) as saved:\n            validate_arrays(saved)\n            if not np.array_equal(saved[\'slices_uint8\'],data[\'slices_uint8\']):raise ValueError(\'Saved uint8 slices mismatch\')\n        with file.open(\'rb\') as stream:digest=hashlib.file_digest(stream,\'sha256\').hexdigest()\n        return dict(study_uid=uid,file=str(file.relative_to(output)).replace(\'\\\\\',\'/\'),split=split_name,\n                    windows=audit[\'windows\'],source_slices=audit[\'source_slices\'],cross_series_windows=0,\n                    slot_present=data[\'slot_present\'].tolist(),bytes=file.stat().st_size,sha256=digest)\n    progress()\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        futures={pool.submit(one,uid):uid for uid in ids}\n        for future in as_completed(futures):\n            try:\n                row=future.result();results.append(row)\n                with (output/\'index.jsonl\').open(\'a\') as stream:stream.write(json.dumps(row)+\'\\n\')\n            except Exception as exc:\n                failure=dict(study_uid=futures[future],error=f\'{type(exc).__name__}: {exc}\');failures.append(failure)\n                with (output/\'failures.jsonl\').open(\'a\') as stream:stream.write(json.dumps(failure)+\'\\n\')\n                print(\'STUDY_FAILED \'+json.dumps(failure),flush=True)\n            progress()\n    manifest=dict(recipe_sha256=RECIPE_SHA,shard=shard,shards=shards,expected_uids=ids,completed=results,\n                  failures=failures,complete=len(results)==len(ids) and not failures)\n    json_write(output/\'manifest.json\',manifest);progress(\'complete\' if manifest[\'complete\'] else \'incomplete\')\n    if not manifest[\'complete\']:raise RuntimeError(\'Incomplete shard; inspect failures.jsonl\')\n    return manifest\n\n\n'
import sys
from pathlib import Path
Path('/kaggle/working/preprocess_six_slot_raptor.py').write_text(BASE_HELPER)
Path('/kaggle/working/preprocess_series_safe_384.py').write_text(PARENT_HELPER)
sys.path.insert(0,'/kaggle/working')


In [ ]:
"""Same pixel contract as series-safe 384; globally minimize triplet reuse."""
from __future__ import annotations
import hashlib
import json
from functools import lru_cache
from pathlib import Path
import numpy as np
import preprocess_series_safe_384 as parent

SLOTS=parent.SLOTS
PARENT_SHA=parent.RECIPE_SHA
BASE_VALIDATE=parent.validate_arrays
TARGET_WEIGHTS=(16,12,10,6,10,6)
RECIPE={**parent.RECIPE,
    'format':'rsna_min_overlap_384_v1','parent_recipe_sha256':PARENT_SHA,
    'intensity_reference':'UNCHANGED parent reference indices: unique linspace over native 2-98% band, using 18/14/12/8/12/8 per selected series',
    'sampling':'up to 180 unique slices; maximize distinct slices used by up to 60 unique same-series windows',
    'source_budget':180,'triplet_budget':60,'slot_triplet_weights':TARGET_WEIGHTS,
    'allocation':'exact dynamic programming: maximize used unique slices, then maximum mutually disjoint subset, then minimize squared slot-weight deviation',
    'triplets':'three adjacent slices in each series sampled list; disjoint groups first, overlapping windows only as required; no duplicate windows',
    'window_fallback':'keep fewer than 60 if selected series have fewer than 60 unique neighboring three-slice windows; no duplicate padding',
    'selected_series':'UNCHANGED one first matching acquired series per six-slot category; no additional series',
    'missing_slots':'absent; triplet budget can shift to available selected slots to minimize reuse; no category substitution',
    'sampler_priority':'minimize repeated source-slice uses at the attainable triplet count, before matching category weights'}
RECIPE_SHA=hashlib.sha256(json.dumps(RECIPE,sort_keys=True).encode()).hexdigest()


def allocate_triplets(capacities,target=60):
    allocation,optimal=_allocate_cached(tuple(int(c) for c in capacities),int(target))
    return allocation.copy(),dict(optimal)


@lru_cache(maxsize=4096)
def _allocate_cached(capacities,target):
    """Exact optimal allocation for <=6 series and a <=60 triplet bag.

    c native eligible slices support c-2 distinct neighboring windows. With k
    windows, min(c,3*k) unique slices and min(k,c//3) disjoint windows are
    simultaneously attainable by series_windows below.
    """
    capacities=tuple(int(c) for c in capacities)
    if len(capacities)!=6 or any(c<0 for c in capacities):raise ValueError('Invalid six-slot capacity')
    target=min(int(target),sum(max(c-2,0) for c in capacities))
    if target<1:raise ValueError('No eligible three-slice window')
    weights=[w if c>=3 else 0 for w,c in zip(TARGET_WEIGHTS,capacities)]
    denominator=sum(weights)
    # State is (unique used, mutually disjoint subset, negative mix penalty).
    states={0:((0,0,0),())}
    for c,w in zip(capacities,weights):
        following={}
        for total,(score,path) in states.items():
            for k in range(min(max(c-2,0),target-total)+1):
                candidate=(score[0]+min(c,3*k),score[1]+min(k,c//3),
                    score[2]-(k*denominator-target*w)**2)
                old=following.get(total+k)
                if old is None or candidate>old[0] or (candidate==old[0] and path+(k,)>old[1]):
                    following[total+k]=(candidate,path+(k,))
        states=following
    score,allocation=states[target]
    return np.asarray(allocation,np.int32),dict(target_triplets=target,optimal_unique_slices=score[0],
        optimal_disjoint_subset=score[1],minimum_repeated_slice_uses=3*target-score[0])


def series_windows(lo,hi,k):
    """Return native sample indices, local windows, and a disjoint-subset mask."""
    c=int(hi-lo+1);k=int(k)
    if k<1 or c<3 or k>c-2:raise ValueError('Invalid series window count')
    count=min(c,3*k)
    picks=np.linspace(lo,hi,count).round().astype(np.int32)
    if len(np.unique(picks))!=count:raise ValueError('Duplicate native sample')
    q=min(k,count//3)
    baseline=set(range(0,3*q,3));starts=set(baseline)
    usage=np.zeros(count,np.int32)
    for start in starts:usage[start:start+3]+=1
    if count%3 and len(starts)<k:
        tail=count-3;starts.add(tail);usage[tail:tail+3]+=1
    while len(starts)<k:
        candidates=[s for s in range(count-2) if s not in starts]
        # All sources are already covered: minimize concentrated reuse.
        start=min(candidates,key=lambda s:(int((usage[s:s+3]+1).max()),
            int(((usage[s:s+3]+1)**2-usage[s:s+3]**2).sum()),s))
        starts.add(start);usage[start:start+3]+=1
    ordered=sorted(starts)
    windows=np.asarray([[s,s+1,s+2] for s in ordered],np.int32)
    primary=np.asarray([s in baseline for s in ordered],bool)
    assert (usage>0).all() and len(windows)==k and primary.sum()==min(k,c//3)
    return picks,windows,primary


def prepare(uid,rows,series_root):
    candidates=[]
    for row in rows:
        candidate=parent.classify(row,series_root/uid/str(row['SeriesInstanceUID']))
        if candidate is not None:candidates.append(candidate)
    used=set();selected=[];capacities=[]
    for slot,(name,plane,fluid,fs,reference_count) in enumerate(SLOTS):
        matches=[s for s in candidates if s['uid'] not in used and s['plane']==plane and
            s['fluid']==fluid and s['fs']==fs and (fluid or s['weight']=='T1')]
        if not matches:selected.append(None);capacities.append(0);continue
        series=matches[0];used.add(series['uid'])
        files=parent.physical_order(series,uid)
        refs=parent.sample_indices(len(files),reference_count)
        lo,hi=int(refs[0]),int(refs[-1])
        selected.append((series,files,refs,lo,hi));capacities.append(hi-lo+1)
    allocation,optimal=allocate_triplets(capacities)
    selection=[];slices=[];slot_ids=[];series_uids=[];sources=[];triplets=[];primary=[];intensity=[]
    for slot,(entry,k) in enumerate(zip(selected,allocation)):
        name,_,_,_,reference_count=SLOTS[slot]
        if entry is None:
            selection.append(dict(slot=name,present=False,triplets=0));continue
        series,files,refs,lo,hi=entry
        details=dict(slot=name,present=True,series_uid=series['uid'],description=series['description'],
            weight=series['weight'],native_depth=len(files),eligible_lo=lo,eligible_hi=hi,
            eligible_count=hi-lo+1,reference_allocation=reference_count,reference_count=len(refs),triplets=int(k))
        if not k:selection.append({**details,'sampled_count':0});continue
        picks,windows,disjoint=series_windows(lo,hi,int(k))
        arrays={int(i):parent.pixels(files[int(i)]['file']) for i in sorted(set(refs)|set(picks))}
        low,high=np.percentile(np.concatenate([arrays[int(i)].ravel() for i in refs]),[2,98])
        offset=len(slices)
        for native_index in picks:
            record=files[int(native_index)]
            slices.append(parent.crop_plane(arrays[int(native_index)],record['spacing'],low,high))
            slot_ids.append(slot);series_uids.append(series['uid'])
            sources.append(dict(slot=slot,series_uid=series['uid'],file=Path(record['file']).name,
                sop_uid=record['sop_uid'],native_index=int(native_index),position=record['position'],spacing=record['spacing']))
        triplets.extend((windows+offset).tolist());primary.extend(disjoint.tolist())
        selection.append({**details,'sampled_count':len(picks),'disjoint_subset':int(disjoint.sum())})
        intensity.append(dict(slot=slot,p2=float(low),p98=float(high),reference_indices=refs.tolist()))
    slices=np.stack(slices);indices=np.asarray(triplets,np.int32);slot_ids=np.asarray(slot_ids,np.int8)
    series_uids=np.asarray(series_uids);usage=np.bincount(indices.ravel(),minlength=len(slices)).astype(np.int8)
    audit=dict(study_uid=uid,recipe_sha256=RECIPE_SHA,parent_recipe_sha256=PARENT_SHA,
        selection=selection,intensity=intensity,source_files=sources,source_slices=len(slices),windows=len(indices),
        cross_series_windows=0,eligible_capacities=capacities,triplet_allocation=allocation.tolist(),
        repeated_slice_uses=int(3*len(indices)-len(slices)),reused_source_slices=int((usage>1).sum()),
        max_slice_usage=int(usage.max()),disjoint_subset_size=int(sum(primary)),**optimal)
    data=dict(slices_uint8=slices,slice_slot_ids=slot_ids,slice_series_uids=series_uids,
        triplet_indices=indices,triplet_series_uids=series_uids[indices[:,0]],
        images=parent.normalized_triplets(slices,indices),triplet_slot_ids=slot_ids[indices],
        same_series=np.ones(len(indices),bool),slot_present=np.asarray([s['present'] for s in selection],bool),
        slice_usage_counts=usage,primary_disjoint_subset=np.asarray(primary,bool),audit_json=np.asarray(json.dumps(audit,allow_nan=False)))
    validate_arrays(data)
    return data,audit


def validate_arrays(data):
    BASE_VALIDATE(data)
    indices=data['triplet_indices'];audit=json.loads(str(data['audit_json']))
    if len(indices)>60 or not np.all(np.diff(indices,axis=1)==1):raise ValueError('Invalid neighboring triplet')
    if len({tuple(row) for row in indices})!=len(indices):raise ValueError('Duplicate triplet')
    usage=np.bincount(indices.ravel(),minlength=len(data['slices_uint8']))
    if not (usage>0).all() or not np.array_equal(usage,data['slice_usage_counts']):raise ValueError('Invalid source reuse counts')
    if len({r['sop_uid'] for r in audit['source_files']})!=len(usage):raise ValueError('Reused or duplicate source SOP')
    allocation,optimal=allocate_triplets(audit['eligible_capacities'])
    if audit['recipe_sha256']!=RECIPE_SHA or audit['triplet_allocation']!=allocation.tolist():raise ValueError('Wrong sampling recipe')
    if len(indices)!=optimal['target_triplets'] or len(usage)!=optimal['optimal_unique_slices']:
        raise ValueError('Non-optimal unique-slice coverage')
    if 3*len(indices)-len(usage)!=optimal['minimum_repeated_slice_uses']:raise ValueError('Avoidable slice reuse')
    primary=data['primary_disjoint_subset']
    if primary.shape!=(len(indices),) or primary.sum()!=optimal['optimal_disjoint_subset']:
        raise ValueError('Non-optimal disjoint subset')
    members=indices[primary].ravel()
    if len(np.unique(members))!=len(members):raise ValueError('Primary subset overlaps')


def run(shard=0,shards=32,limit=None,root=None,output=None,workers=4):
    # Reuse the exact label, Gold74, save/reopen/checksum and progress machinery.
    # Only this notebook process's imported parent module is configured.
    parent.prepare=prepare;parent.validate_arrays=validate_arrays
    parent.RECIPE=RECIPE;parent.RECIPE_SHA=RECIPE_SHA
    return parent.run(shard=shard,shards=shards,limit=limit,root=root,
        output=output or f'/kaggle/working/min_overlap_384_shard_{shard:02d}',workers=workers)


In [ ]:
manifest=run(shard=24,shards=32,limit=None,workers=4)
assert manifest['complete']
